# **MLB Stuff+ Solution Pipeline**

### **This notebook contains the project problem solution pipeline. The data collected using the data generating scripts will be loaded into a DuckDB database, and used to solve the project problem:**

Using MLB Statcast Data for the 2021-2025 Seasons and Expected Run Values, design a machine learning model to predict the performance of MLB Pitchers based on the quality of their individual pitches in terms of their expected run value, standardized into a normally distributed `Stuff+` value, validating with next-season statistical outcomes.

### **Pipeline Contents**

- Package Imports
- Logging
- Loading Data into DuckDB Database
- Querying Database into Pandas DataFrames
- Pre-Training Feature Engineering
- Model Training
- Model Prediction
- Visualization of Results

## **Imports**

The following packages are imported for use in the solution pipeline.

In [1]:
# Imports
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import logging
import duckdb

## **Logger Initialization**

This is the initialization of the logger used throughout the pipeline notebook.

In [2]:
# Initializing logger
logging.basicConfig(
    level = logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',  # Timestamp, level, and message format
    filename = 'pipeline.log'                            # Log output file
)
logger = logging.getLogger(__name__)  # Create a logger instance

## **Loading Data into DuckDB Database**

Within the following code chunks, each of the 4 data files are loaded directly into a centralized DuckDB Database.

In [3]:
# Connect to and/or Create DuckDB Database
try:
    con = duckdb.connect(database = 'stuff-plus.db', read_only = False)
    print("Connected to DuckDB instance")
    logger.info("Connected to DuckDB instance")
except Exception as e:
    print(f"Failed to connect to DuckDB instance: {e}")
    logger.error(f"Failed to connect to DuckDB instance: {e}")

# Remove existing tables if they exist from a previous run of the code
try:
    con.execute(f"""
        -- Drop the StatcastPitch table if it exists
        DROP TABLE IF EXISTS StatcastPitch;
                
        -- Drop the PitcherStats table
        DROP TABLE IF EXISTS PitcherStats;
                
        -- Drop the PitcherBio table if it exists
        DROP TABLE IF EXISTS PitcherBio;
                
        -- Drop the ExpectedRunValue table
        DROP TABLE IF EXISTS ExpectedRunValue;   
        """)
    print("Dropped tables if they already existed")
    logger.info("Dropped tables if they already existed")
except Exception as e:
    print(f"Failed to drop existing tables: {e}")
    logger.error(f"Failed to drop existing tables: {e}")

# Load the data files into the Database
try:
    # Loading in the StatcastPitch data
    con.execute(f"""
        -- Create the StatcastPitch table
                CREATE TABLE StatcastPitch
                AS
                Select * FROM read_parquet('data/statcast-data.parquet');
        """)
    print("Loaded StatcastPitch Table")
    logger.info("Loaded StatcastPitch Table")

    # Loading in the PitcherStats data
    con.execute(f"""
        -- Create the PitcherStats table
                CREATE TABLE PitcherStats
                AS
                Select * FROM read_parquet('data/pitcher-stats.parquet');
        """)
    print("Loaded PitcherStats Table")
    logger.info("Loaded PitcherStats Table")

    # Loading in the PitcherBio data
    con.execute(f"""
        -- Create the PitcherBio table
                CREATE TABLE PitcherBio
                AS
                Select * FROM read_parquet('data/pitcher-bio.parquet');
        """)
    print("Loaded PitcherBio Table")
    logger.info("Loaded PitcherBio Table")

    # Loading in the ExpectedRunValue data
    con.execute(f"""
        -- Create the ExpectedRunValue table
                CREATE TABLE ExpectedRunValue
                AS
                Select * FROM read_parquet('data/run-values.parquet');
        """)
    print("Loaded ExpectedRunValue Table")
    logger.info("Loaded ExpectedRunValue Table")
except Exception as e:
    print(f"Failed to load data into DuckDB: {e}")
    logger.error(f"Failed to load data into DuckDB: {e}")

Connected to DuckDB instance
Dropped tables if they already existed


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded StatcastPitch Table
Loaded PitcherStats Table
Loaded PitcherBio Table
Loaded ExpectedRunValue Table


## **Querying Database into Pandas DataFrames**

Within the following code chunks, the data tables will be queried out of the DuckDB database into Pandas DataFrames, which will be used for the modeling/analysis stage. The following DataFrames will be queried:

- **model_df:** A join between the StatcastPitch and ExpectedRunValue tables, to be used for feature engineering and model training
- **bio_df**: The PitcherBio table, to be used for analysis/visualization
- **stats_df**: The PitcherStats table, to be used for analysis/visualization

In [ ]:
# Query tables into pandas dataframes
try:
    # Query model_df DataFrame
    model_df = con.execute("""
        SELECT 
            s.*,                                                    -- All columns of StatcastPitch
            e.delta_run_exp AS target                               -- delta_run_exp from ExpectedRunValue as the model target variable
        FROM StatcastPitch s
        LEFT JOIN ExpectedRunValue e
            ON s.event_balls_strikes = e.event_balls_strikes;       -- Left join on the event_balls_strikes column
    """).fetchdf()
    print(f"Extracted model_df")
    logger.info(f"Extracted model_df")

    # Query bio_df DataFrame
    bio_df = con.execute("""
        SELECT * FROM PitcherBio;
        """).fetchdf()
    print(f"Extracted bio_df")
    logger.info(f"Extracted bio_df")

    # Query stats_df DataFrame
    stats_df = con.execute("""
        SELECT * FROM PitcherStats;
        """).fetchdf()
    print(f"Extracted stats_df")
    logger.info(f"Extracted stats_df")

except Exception as e:
    print(f"Failed to query tables into DataFrames: {e}")
    logger.error(f"Failed to query tables into DataFrames: {e}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Extracted model_df
Extracted bio_df
Extracted stats_df


## **Pre-Training Feature Engineering**

Within the following code chunks, feature engineering will be conducted. Explanations/justifications for the feature engineering decisions are included in the corresponding markdown chunks.